# 02 - Ahora armaremos trasnformamos y hacemos ingenieria de caracteristicas

In [1]:
from rich import default_styles
from rich import default_styles
import pandas as pd
import sys
import os
from pathlib import Path

# Subir un nivel a la raíz del proyecto y agregarla a sys.path
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJ_ROOT)
if str(PROJ_ROOT) not in sys.path:
    sys.path.append(str(PROJ_ROOT))

from src.features.build_features import construir_target

df_limpio = pd.read_parquet("data/processed/dataset_modelado.parquet")
print(f"Filas orginales : {len(df_limpio)}")
df_limpio.head(10)

Filas orginales : 3965


,id_alumno,anio_academico,cohorte_carrera,cohorte_universidad,cambio_plan,antiguedad_carrera,antiguedad_universidad,anios_antes_ingreso_carrera,tuvo_actividad_previa,anios_actividad_previa,...,tasa_desaprobacion_finales,tasa_ausencias_finales,equivalencias_asignaturas,asignaturas_aprobadas_acumuladas,equivalencias_finales,finales_aprobados_acumulados,porcentaje_avance_cursado,porcentaje_avance_carrera,cantidad_pausas,maxima_duracion_pausa
0,1,2026,2026,2026,0,0,0,0,0,0,...,0.0,0.00,2,2,2,2,0.042,0.045,0,0
1,2,2023,2022,2022,1,1,1,0,0,0,...,0.0,0.00,0,0,0,0,0.000,0.000,0,0
2,3,2003,2003,2003,1,0,0,0,0,0,...,0.0,1.00,4,5,4,4,0.104,0.091,0,0
3,3,2004,2003,2003,1,1,1,0,0,0,...,0.0,0.00,4,5,4,4,0.104,0.091,0,0
4,3,2005,2003,2003,1,2,2,0,0,0,...,0.0,0.67,4,5,4,5,0.104,0.114,0,0
5,4,2008,2008,2008,1,0,0,0,0,0,...,0.0,0.00,0,0,0,0,0.000,0.000,0,0
6,5,2021,2021,1996,1,0,25,25,0,0,...,0.0,0.00,0,0,0,0,0.000,0.000,0,0
7,6,2021,2021,2021,1,0,0,0,1,3,...,0.0,0.00,7,11,6,10,0.229,0.227,1,21
8,6,2022,2021,2021,1,1,1,0,1,3,...,0.0,0.00,7,15,6,14,0.313,0.318,1,21
9,6,2023,2021,2021,1,2,2,0,1,3,...,0.0,0.00,7,16,6,15,0.333,0.341,1,21


In [2]:
df_con_target = construir_target(df_limpio)

print(f"Filas tras definir target y censurar últimas cohortes: {len(df_con_target)}")
df_con_target[["id_alumno", "anio_academico", "prox_actividad", "abandono"]].head(10)


Filas tras definir target y censurar últimas cohortes: 1979


,id_alumno,anio_academico,prox_actividad,abandono
1,2,2023,NaN,1.0
2,3,2003,2004.0,0.0
3,3,2004,2005.0,0.0
4,3,2005,NaN,1.0
5,4,2008,NaN,1.0
6,5,2021,NaN,1.0
7,6,2021,2022.0,0.0
8,6,2022,2023.0,0.0
9,6,2023,2025.0,0.0
12,7,2021,2022.0,0.0


In [3]:
from IPython.display import display  

import numpy as np

# Tomamos 5 alumnos aleatorios
np.random.seed(42)
alumnos_muestra = np.random.choice(df_con_target["id_alumno"].unique(), size=5, replace=False)

for id_estudiante in alumnos_muestra:
    print(f"\n--- Historial del Alumno {id_estudiante} ---")
    sub = df_con_target[df_con_target["id_alumno"] == id_estudiante][
        ["id_alumno", "anio_academico", "prox_actividad", "abandono"]
    ]
    display(sub)



--- Historial del Alumno 552 ---


,id_alumno,anio_academico,prox_actividad,abandono
1783,552,2021,2022.0,0.0
1784,552,2022,2023.0,0.0
1785,552,2023,2024.0,0.0



--- Historial del Alumno 481 ---


,id_alumno,anio_academico,prox_actividad,abandono
1565,481,2021,2022.0,0.0
1566,481,2022,2023.0,0.0
1567,481,2023,NaN,1.0



--- Historial del Alumno 467 ---


,id_alumno,anio_academico,prox_actividad,abandono
1536,467,2021,NaN,1.0



--- Historial del Alumno 36 ---


,id_alumno,anio_academico,prox_actividad,abandono
116,36,2004,2005.0,0.0
117,36,2005,NaN,1.0



--- Historial del Alumno 647 ---


,id_alumno,anio_academico,prox_actividad,abandono
2064,647,2022,NaN,1.0


In [4]:
conteo = df_con_target["abandono"].value_counts()
proporcion = df_con_target["abandono"].value_counts(normalize=True) * 100

print("Distribución de la variable objetivo:")
print(f" - Continúan (0): {conteo[0]} ({proporcion[0]:.2f}%)")
print(f" - Abandonaron (1): {conteo[1]} ({proporcion[1]:.2f}%)")
print(f"Ratio de desbalance: 1 a {conteo[0] / conteo[1]:.1f}")


Distribución de la variable objetivo:
 - Continúan (0): 1380 (69.73%)
 - Abandonaron (1): 599 (30.27%)
Ratio de desbalance: 1 a 2.3


- Como se puede ver 

In [5]:
df_con_target.to_parquet("data/processed/dataset_modelado.parquet", index=False)
print("Dataset actualizado con variable objetivo.")


Dataset actualizado con variable objetivo.
